# 03 — SQL analysis

This notebook executes all 40 portfolio queries in thematic sections. Each cell shows
up to ten rows; the SQL file remains the source of truth, and the full catalog provides
the interpretation and methodology.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "sql" / "create_tables.sql").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing sql/create_tables.sql")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATABASE_PATH = PROJECT_ROOT / "instacart.duckdb"
PROJECT_ROOT

PosixPath('/Users/sebastiankalita/Documents/DE_portfolio/sql-project/instacart-sql-analysis')

In [2]:
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        "Database not found. Run 01_load_and_explore.ipynb or "
        "python scripts/build_database.py first."
    )

query_files = {
    int(path.name[:2]): path
    for path in (PROJECT_ROOT / "sql").glob("*/*.sql")
    if path.name[:2].isdigit()
}
assert sorted(query_files) == list(range(1, 41)), "Expected query IDs 01-40 exactly once"

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)


def run_query(query_id: int, preview_rows: int = 10) -> pd.DataFrame:
    query = query_files[query_id].read_text().strip().rstrip(";")
    preview_query = f"SELECT * FROM ({query}) AS portfolio_query LIMIT {preview_rows}"
    return connection.execute(preview_query).df()

## Dataset overview

### Query 01 — How many unique users are included in the dataset?

In [3]:
run_query(1)

,users_count
0,206209


### Query 02 — How many orders have been placed?

In [4]:
run_query(2)

,orders_count
0,3421083


### Query 03 — How many products belong to each department?

In [5]:
run_query(3)

,department,product_count
0,personal care,6563
1,snacks,6264
2,pantry,5371
3,beverages,4365
4,frozen,4007
5,dairy eggs,3449
6,household,3085
7,canned goods,2092
8,dry goods pasta,1858
9,produce,1684


## Product performance

### Query 04 — Which products are purchased most frequently?

In [6]:
run_query(4)

,product_id,name,purchase_count
0,24852,Banana,491291
1,13176,Bag of Organic Bananas,394930
2,21137,Organic Strawberries,275577
3,21903,Organic Baby Spinach,251705
4,47209,Organic Hass Avocado,220877
5,47766,Organic Avocado,184224
6,47626,Large Lemon,160792
7,16797,Strawberries,149445
8,26209,Limes,146660
9,27845,Organic Whole Milk,142813


### Query 05 — Which products have the fewest recorded purchases?

In [7]:
run_query(5)

,name,purchase_count
0,Yellow Fish Breading,1
1,12 Inch Taper Candle White,1
2,Greek Blended Cherry Fat Free Yogurt,1
3,Berry Sprouted Blend Cereal,1
4,Water With Electrolytes,1
5,Natural Peanut Butter With Flaxseed,1
6,Petite Black Eyed Peas,1
7,Organic Better Rest Tea Blend,1
8,Easter Basket,1
9,Wasabi Cheddar Spreadable Cheese,1


### Query 06 — Which departments account for the most purchases?

In [8]:
run_query(6)

,department_id,department,purchase_count
0,4,produce,9888378
1,16,dairy eggs,5631067
2,19,snacks,3006412
3,7,beverages,2804175
4,1,frozen,2336858


### Query 07 — Which aisles have the highest purchase volume?

In [9]:
run_query(7)

,aisle_name,sales_volume
0,fresh fruits,3792661
1,fresh vegetables,3568630
2,packaged vegetables fruits,1843806
3,yogurt,1507583
4,packaged cheese,1021462


### Query 08 — Which products are most frequently added to the cart first?

In [10]:
run_query(8)

,product_id,product_name,product_count
0,24852,Banana,115521
1,13176,Bag of Organic Bananas,82877
2,27845,Organic Whole Milk,32071


### Query 09 — Which products are most frequently added to the cart last?

In [11]:
run_query(9)

,ID,product_name,product_count
0,24852,Banana,31140
1,13176,Bag of Organic Bananas,30770
2,21137,Organic Strawberries,24359


### Query 10 — What are the top three products within every department?

In [12]:
run_query(10)

,department,product_name,product_count,product_rank
0,alcohol,Sauvignon Blanc,8541,1
1,alcohol,Cabernet Sauvignon,6352,2
2,alcohol,Chardonnay,6346,3
3,babies,Baby Food Stage 2 Blueberry Pear & Purple Carrot,9103,1
4,babies,Spinach Peas & Pear Stage 2 Baby Food,8303,2
5,babies,Gluten Free SpongeBob Spinach Littles,7342,3
6,bakery,100% Whole Wheat Bread,63114,1
7,bakery,Organic Bread with 21 Whole Grains,23944,2
8,bakery,Ezekiel 4:9 Bread Organic Sprouted Whole Grain,18487,3
9,beverages,Sparkling Water Grapefruit,79245,1


### Query 11 — What are the ten highest-ranked products by purchases?

In [13]:
run_query(11)

,product_name,product_count,ranking
0,Banana,491291,1
1,Bag of Organic Bananas,394930,2
2,Organic Strawberries,275577,3
3,Organic Baby Spinach,251705,4
4,Organic Hass Avocado,220877,5
5,Organic Avocado,184224,6
6,Large Lemon,160792,7
7,Strawberries,149445,8
8,Limes,146660,9
9,Organic Whole Milk,142813,10


### Query 12 — Which products were purchased by more than 10% of users?

In [14]:
run_query(12)

,product_id,product_name,users_rate
0,24852,Banana,0.37
1,13176,Bag of Organic Bananas,0.32
2,21137,Organic Strawberries,0.30
3,21903,Organic Baby Spinach,0.28
4,47626,Large Lemon,0.24
5,26209,Limes,0.23
6,47209,Organic Hass Avocado,0.22
7,16797,Strawberries,0.22
8,47766,Organic Avocado,0.21
9,39275,Organic Blueberries,0.19


### Query 13 — Which department has the widest purchased product variety?

In [15]:
run_query(13)

,department_id,department,distinct_product_id
0,11,personal care,6563


### Query 14 — How do departments compare across key performance measures?

In [16]:
run_query(14)

,department,products_num,sales,reorder_rate,sales_prcnt
0,produce,1684,9888378,0.650521,0.292390
1,dairy eggs,3449,5631067,0.670161,0.166505
2,snacks,6264,3006412,0.574464,0.088897
3,beverages,4365,2804175,0.653651,0.082917
4,frozen,4007,2336858,0.542634,0.069099
5,pantry,5371,1956819,0.347400,0.057861
6,bakery,1516,1225181,0.628381,0.036227
7,canned goods,2092,1114857,0.458639,0.032965
8,deli,1322,1095540,0.608130,0.032394
9,dry goods pasta,1858,905340,0.462220,0.026770


### Query 15 — What are the top five products per department, including ties?

In [17]:
run_query(15)

,department,product_id,product_name,sales_in_dept,ranking
0,alcohol,2120,Sauvignon Blanc,8541,1
1,alcohol,33065,Cabernet Sauvignon,6352,2
2,alcohol,38444,Chardonnay,6346,3
3,alcohol,46088,Beer,6068,4
4,alcohol,45190,Vodka,5666,5
5,babies,43875,Baby Food Stage 2 Blueberry Pear & Purple Carrot,9103,1
6,babies,34134,Spinach Peas & Pear Stage 2 Baby Food,8303,2
7,babies,2611,Gluten Free SpongeBob Spinach Littles,7342,3
8,babies,3020,Broccoli & Apple Stage 2 Baby Food,7054,4
9,babies,44471,Free & Clear Unscented Baby Wipes,6540,5


## Basket behaviour

### Query 16 — What is the average basket size?

In [18]:
run_query(16)

,avg_basket_size
0,10.107073


### Query 17 — What is the largest observed basket?

In [19]:
run_query(17)

,max_basket_size
0,145


### Query 18 — How are basket sizes distributed?

In [20]:
run_query(18)

,basket_size,distribution
0,1,163593
1,2,194361
2,3,215060
3,4,230299
4,5,237225
5,6,236383
6,7,228547
7,8,211357
8,9,191564
9,10,172103


### Query 19 — How many department items appear in an order containing that department?

In [21]:
run_query(19)

,department_name,avg_department_items_per_order
0,produce,3.95
1,dairy eggs,2.49
2,babies,2.38
3,snacks,2.08
4,frozen,1.90
5,beverages,1.85
6,alcohol,1.81
7,pantry,1.68
8,pets,1.65
9,household,1.57


### Query 20 — How does each basket compare with the previous observed basket?

In [22]:
run_query(20)

,user_id,order_id,order_number,basket_size,basket_size_change
0,1,2539329,1,5,5
1,1,2398795,2,6,1
2,1,473747,3,5,-1
3,1,2254736,4,5,0
4,1,431534,5,8,3
5,1,3367565,6,4,-4
6,1,550135,7,5,1
7,1,3108588,8,6,1
8,1,2295261,9,6,0
9,1,2550362,10,9,3


### Query 21 — Which users increased basket size at every observed order?

In [23]:
run_query(21)

,user_id,orders_count,min_basket_size_change
0,416,4,1
1,659,3,2
2,1057,3,12
3,1445,3,4
4,1765,3,1
5,2014,4,2
6,2324,3,3
7,3508,4,2
8,4126,3,1
9,5318,3,1


### Query 22 — What is the median basket size?

In [24]:
run_query(22)

,median_basket_size
0,8.0


### Query 23 — What is each user's three-order rolling average basket size?

In [25]:
run_query(23)

,user_id,order_number,order_id,basket_size,moving_avg_basket_size
0,1,1,2539329,5,5.000000
1,1,2,2398795,6,5.500000
2,1,3,473747,5,5.333333
3,1,4,2254736,5,5.333333
4,1,5,431534,8,6.000000
5,1,6,3367565,4,5.666667
6,1,7,550135,5,5.666667
7,1,8,3108588,6,5.000000
8,1,9,2295261,6,5.666667
9,1,10,2550362,9,7.000000


## Customer behaviour

### Query 24 — Which users placed the most orders?

In [26]:
run_query(24)

,user_id,number_of_orders
0,132863,100
1,170771,100
2,82232,100
3,171544,100
4,130465,100
5,129784,100
6,83161,100
7,169608,100
8,83011,100
9,131864,100


### Query 25 — How are users distributed across order-frequency segments?

In [27]:
run_query(25)

,order_segment,users_count
0,1-5 orders,43576
1,6-10 orders,60937
2,11-20 orders,50965
3,More than 20 orders,50731


### Query 26 — Which users belong to the top 10% by order count?

In [28]:
run_query(26)

,user_id,order_count,percent_rank
0,10747,100,0.0
1,178113,100,0.0
2,66482,100,0.0
3,182364,100,0.0
4,106161,100,0.0
5,18412,100,0.0
6,158697,100,0.0
7,139660,100,0.0
8,22337,100,0.0
9,93062,100,0.0


### Query 27 — Can order sequence be recreated with a window function?

In [29]:
run_query(27)

,user_id,order_id,generated_order_number
0,126,1747819,1
1,126,1020546,2
2,126,1480579,3
3,126,1537356,4
4,126,1107715,5
5,126,1879545,6
6,126,2117732,7
7,126,33119,8
8,126,2452141,9
9,126,32138,10


### Query 28 — Which customers average fewer than seven days between orders?

In [30]:
run_query(28)

,user_id,avg_days_between_orders,orders_count
0,14433,0.000000,7
1,74329,0.000000,6
2,80567,0.000000,4
3,109010,0.000000,4
4,164320,0.000000,4
5,201321,0.000000,4
6,138747,0.166667,7
7,46408,0.232323,100
8,168850,0.250000,5
9,134572,0.292929,100


### Query 29 — Which user has the strongest concentration in one department?

In [31]:
run_query(29)

,user_id,department_id,total_purchases,loyalty_rate
0,201038,4,530.0,1.0


### Query 30 — What is the estimated observed lifetime and order count per user?

In [32]:
run_query(30)

,user_id,orders_count,days_between_first_and_last_order
0,30633,64,365.0
1,53474,61,365.0
2,55673,59,365.0
3,55649,44,365.0
4,53897,57,365.0
5,56214,39,365.0
6,31004,88,365.0
7,57109,72,365.0
8,29932,64,365.0
9,57177,40,365.0


## Reorder analysis

### Query 31 — Which department has the highest reorder rate?

In [33]:
run_query(31)

,dept_ID,dept_name,reorder_rate
0,16,dairy eggs,67.02


### Query 32 — What percentage of purchased items are reorders?

In [34]:
run_query(32)

,reorder_rate
0,59.01


### Query 33 — Which aisles have the highest reorder rates?

In [35]:
run_query(33)

,aisle_name,reorder_rate
0,milk,78.18
1,water seltzer sparkling water,72.99
2,fresh fruits,71.88


### Query 34 — Which department generates the most reordered purchases?

In [36]:
run_query(34)

,department,reordered_purchase_count
0,produce,6432596


### Query 35 — Which product or tied products lead reorder rate by department?

In [37]:
run_query(35)

,department,product_name,reorder_rate
0,bakery,Wheat Sandwich Bread,0.828140
1,personal care,Serenity Ultimate Extrema Overnight Pads,0.933333
2,meat seafood,Bockwurst,0.785714
3,breakfast,Hot & Fit Mayan Blend Cereal,0.833333
4,bulk,Dried Mango,0.745847
5,alcohol,Russian River Valley Reserve Pinot Noir,0.900000
6,deli,Raw Veggie Wrappers,0.942029
7,dairy eggs,Half And Half Ultra Pasteurized,0.861436
8,pets,Fragrance Free Clay with Natural Odor Eliminat...,0.870229
9,other,Simply Sleep Nighttime Sleep Aid,0.911111


### Query 36 — Which sufficiently popular product has the highest reorder rate?

In [38]:
run_query(36)

,product_id,product_name,product_count,reorder_rate
0,9292,Half And Half Ultra Pasteurized,2995,0.861436


## Market basket analysis

### Query 37 — Which product pairs occur most often in the order-ID sample below 1,000,000?

In [39]:
run_query(37)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_1_id,product_1_name,product_2_id,product_2_name,pair_count
0,13176,Bag of Organic Bananas,21137,Organic Strawberries,18968
1,13176,Bag of Organic Bananas,47209,Organic Hass Avocado,18818
2,21137,Organic Strawberries,24852,Banana,17030
3,24852,Banana,47766,Organic Avocado,16109
4,21903,Organic Baby Spinach,24852,Banana,15605
5,13176,Bag of Organic Bananas,21903,Organic Baby Spinach,15274
6,16797,Strawberries,24852,Banana,12657
7,24852,Banana,47626,Large Lemon,12571
8,21137,Organic Strawberries,47209,Organic Hass Avocado,12378
9,13176,Bag of Organic Bananas,27966,Organic Raspberries,12301


### Query 38 — Which triples occur most often in the order-ID sample below 10,000?

In [40]:
run_query(38)

,product_1_id,product_1_name,product_2_id,product_2_name,product_3_id,product_3_name,triple_count
0,13176,Bag of Organic Bananas,27966,Organic Raspberries,47209,Organic Hass Avocado,42
1,13176,Bag of Organic Bananas,21137,Organic Strawberries,47209,Organic Hass Avocado,36
2,21137,Organic Strawberries,21903,Organic Baby Spinach,24852,Banana,33
3,13176,Bag of Organic Bananas,21137,Organic Strawberries,21903,Organic Baby Spinach,32
4,13176,Bag of Organic Bananas,21903,Organic Baby Spinach,47209,Organic Hass Avocado,32
5,21137,Organic Strawberries,24852,Banana,47209,Organic Hass Avocado,32
6,5876,Organic Lemon,13176,Bag of Organic Bananas,47209,Organic Hass Avocado,29
7,13176,Bag of Organic Bananas,21137,Organic Strawberries,27966,Organic Raspberries,27
8,24852,Banana,26209,Limes,47766,Organic Avocado,26
9,21903,Organic Baby Spinach,24852,Banana,47766,Organic Avocado,24


### Query 39 — Which products most often appear consecutively in add-to-cart order?

In [41]:
run_query(39)

,before_product_id,before_product_name,next_product_id,next_product_name,pair_count
0,13176,Bag of Organic Bananas,47209,Organic Hass Avocado,9940
1,24852,Banana,47766,Organic Avocado,9540
2,13176,Bag of Organic Bananas,21137,Organic Strawberries,8191
3,24852,Banana,21137,Organic Strawberries,7713
4,47209,Organic Hass Avocado,13176,Bag of Organic Bananas,7411
5,47626,Large Lemon,26209,Limes,7265
6,24852,Banana,28204,Organic Fuji Apple,7068
7,47766,Organic Avocado,24852,Banana,6599
8,21137,Organic Strawberries,13176,Bag of Organic Bananas,6492
9,13176,Bag of Organic Bananas,21903,Organic Baby Spinach,5858


### Query 40 — Which products most often occur in banana-containing baskets?

In [42]:
run_query(40)

,product_id,product_name,product_count
0,21137,Organic Strawberries,134535
1,21903,Organic Baby Spinach,113709
2,47209,Organic Hass Avocado,106051
3,47766,Organic Avocado,85254
4,27966,Organic Raspberries,72327


## Close the read-only connection

In [43]:
connection.close()